In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/dhotepatil00@gmail.com/regis-healthcare/1_setup/utility

In [0]:
print(bronze_schema,silver_schema,gold_schema) 

In [0]:
dbutils.widgets.text("catalog","regis_healthcare","catalog")
dbutils.widgets.text("data_source","facilities","data_source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

#### Silver Processing

In [0]:
df_bronze = spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source};")
display(df_bronze)
print(df_bronze.count())

In [0]:
# schema check
print(df_bronze.count())
df_bronze.printSchema()

In [0]:
df_bronze.columns

In [0]:
# drop duplicate
df_silver = df_bronze.dropDuplicates()
print(df_silver.count())

In [0]:
df_silver = df_silver.withColumn(
    "facility_id",
    F.trim(F.col("facility_id"))
).withColumn(
    "facility_name",
    F.trim(F.col("facility_name"))
).withColumn(
    "address",
    F.trim(F.col("address"))
).withColumn(
    "suburb",
    F.trim(F.col("suburb"))
).withColumn(
    "state",
    F.trim(F.col("state"))
).withColumn(
    "postcode",
    F.trim(F.col("postcode"))
).withColumn(
    "phone",
    F.trim(F.col("phone"))
).withColumn(
    "email",
    F.trim(F.col("email"))
).withColumn(
    "capacity",
    F.trim(F.col("capacity"))
).withColumn(
    "accreditation_status",
    F.trim(F.col("accreditation_status"))
).withColumn(
    "created_at",
    F.trim(F.col("created_at"))
)

In [0]:
# null records count 
from pyspark.sql.functions import col,count,when
null_count = df_silver.select([count(when(col(c).isNull(),c)).alias(c)for c in df_silver.columns
                               ])
display(null_count)

#### Cleaning data in table

In [0]:
# facility_id 
from pyspark.sql.functions import col,when
df_filt = df_silver.filter(~col("facility_id").rlike("^FAC"))

df_silver = df_silver.withColumn(
    "facility_id",
    when(
        (col("facility_id").isNull()) | (~col("facility_id").rlike("^FAC")),
        "0"
    ).otherwise(col("facility_id"))
)

display(df_filt)
display(df_silver)

In [0]:
# facility_name
from pyspark.sql.functions import col,when,initcap,trim
df_silver = df_silver.withColumn("facility_name",initcap(trim(col("facility_name"))))
# display(df_silver)
df_silver = df_silver.withColumn("facility_name",when(col("facility_name").isNull(),"Not Provide").otherwise(col("facility_name")))

df_filt = df_silver.filter(col("facility_id").rlike("^Regis"))
display(df_filt)
display(df_silver)

In [0]:
# address
from pyspark.sql.functions import col,when,initcap,trim
df_silver = df_silver.withColumn("address",initcap(trim(col("address"))))
# display(df_silver)
df_silver = df_silver.withColumn("address",when(col("address").isNull(),"Not Provide").otherwise(col("address")))
display(df_silver)

In [0]:
# suburb
from pyspark.sql.functions import col,when,trim,upper
dup = df_silver.groupBy("suburb").count()
# display(dup)

df_silver = df_silver.withColumn("suburb",upper(trim(col("suburb"))))
display(df_silver)

In [0]:
# state
from pyspark.sql.functions import col,when,trim,upper
dup = df_silver.groupBy("state").count()
# display(dup)

df_silver = df_silver.withColumn("state",upper(trim(col("state"))))
# display(df_silver)
dup = df_silver.groupBy("state").count()
display(dup)

In [0]:
# postcode
from pyspark.sql.functions import col,when,trim,upper
df_invalid = df_silver.filter(col("postcode").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"))
display(df_invalid)

In [0]:
# phone
from pyspark.sql.functions import col,when,trim,upper
# df_invalid = df_silver.filter(~col("phone").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"))
df_filt = df_silver.filter(col("phone").rlike("^[0-9]+$"))
# df_filt = df_silver.filter(col("phone").rlike("^[a-zA-Z]+$"))
display(df_filt)

In [0]:
# email
from pyspark.sql.functions import col,when,trim,upper
# df_invalid = df_silver.filter(~col("email").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"))
# df_filt = df_silver.filter(col("phone").rlike("^[0-9]+$"))
df_filt = df_silver.filter(col("phone").rlike("^[a-zA-Z]+$"))
display(df_filt)

In [0]:
# capacity
df_silver = df_silver.withColumn("capacity",when(col("capacity")< 0,0).otherwise(col("capacity")))
display(df_silver)

In [0]:
# accreditation_status
df_silver = df_silver.withColumn("accreditation_status",initcap(trim(col("accreditation_status"))))
display(df_silver)

In [0]:
# created_at
from pyspark.sql.functions import col, to_timestamp
df_silver = df_silver.withColumn(
    "created_at",
    to_timestamp(col("created_at"), "yyyy-MM-dd HH:mm:ss")  # specify format if needed
)
display(df_silver)

#### Silver table load

In [0]:
df_silver.write\
    .format("delta")\
        .option("delta.enableChangeDataFeed","true")\
            .option("mergeSchema","true")\
                .option("overwriteSchema","true")\
            .mode("overwrite")\
               .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

dt = spark.sql(f"select * from {catalog}.{silver_schema}.{data_source};")
print(dt.count())
display(dt)

In [0]:
# load to s3
df_silver.write.format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
    .mode("overwrite")\
    .partitionBy("current_date")\
    .save(f"s3://regis-healthcare/silver-clean-data/{data_source}/")